**Cell 1**

# NB09.1 — Reward-Free Geometry of the RS-PPO Adapters

Computes the geometry of the five RS-PPO LoRA specialists produced by NB08. This notebook is the **reward-free** half of the analysis pipeline: it never loads ArmoRM and never issues a reward query. Everything here is a statement about where the five task vectors sit relative to each other, independent of how they were trained.

**Scope — what this notebook delivers.**

| Step | Object | Deliverable |
|---|---|---|
| D | $D_i = \theta_i - \theta_{\mathrm{SFT}}$, exact effective LoRA update | prerequisite |
| R | $R = D^\top D$ (Gram) and $R_{\cos}$ | prerequisite |
| Floor-LP, $R^-$ | is the improvement floor $\mathcal F_p$ open? | **(b)** |
| LMC | loss barriers along interpolation paths | **(d)** |

**What this notebook does NOT do.** No ArmoRM, no search set scoring, no vertex matrix $M$, no $U_p(\lambda)$, no method comparison. Those belong to deliverable (a), which is currently blocked pending the §6 escalation. Deliverables (b) and (d) are purely geometric and are unaffected by that escalation — this is exactly why they run first.

**Why not PEFT.** `peft.add_weighted_adapter(..., combination_type="linear")` does **not** realise $\theta_0 + \sum_i \lambda_i \delta_i$: it combines the factors under a square root and the product $B_{\mathrm{merged}}A_{\mathrm{merged}}$ picks up cross terms $B_iA_j$ for every interior $\lambda$. All merging in this notebook goes through `src/merge.py`, which accumulates the effective deltas directly. Cell 20 measures the actual error for this adapter bundle against an independent oracle **and** against a deliberately contaminated control that must fail.

**Precision.** Every merge and every evaluation forward pass runs in **float32**. `MERGE_DTYPE = "float32"` is asserted by static checks S9/S10 before any geometry is computed, so reduced-precision rounding cannot silently enter the measured endpoint displacements.

**NB09.1r5 corrections.** This version preserves the exact positive-$R$ and floor taxonomy from r4 and replaces the uncorrected any-of-ten LMC decision with shared-resample, familywise simultaneous max-deviation intervals. Detection and practical equivalence are separate claims. A positive LMC claim is disabled until the scientific margin $\delta$ is explicitly pre-registered. The analysis still uses all 13 preferences, 256 held-out responses, a 21-point path grid, and full provenance in `report.json`.

**Positive-$R$ policy.** Positive off-diagonal entries are retained as the measured result; the notebook never manufactures sign conflicts by centering or re-signing $R$. It reports sign-conflict absence, floor collapse, and tangent-centred relative geometry as three distinct objects.

**Provisional numbers.** Every floor / $R^-$ / Perron figure currently in the thesis comes from the **v5 SFT-R** and is provisional. This notebook replaces them. Do not assume the floor collapse reproduces — it may or may not.

**Cell 2**

## 1. Clone or update the repository

In [ ]:
# Cell 3
%cd /content
import os, shutil
repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

**Cell 4**

## 2. Check the machine

Unlike NB08 this notebook is mostly **RAM**-bound, not VRAM-bound. The effective delta $B_iA_i$ of one adapter materialises to roughly the full target-weight footprint (~0.97 B parameters for TinyLlama's seven LoRA modules), i.e. ~3.9 GB in fp32 per adapter. R is therefore computed **pairwise streaming** — never more than two adapters resident at once (~8 GB peak) — rather than by holding all five.

A GPU is only needed for the LMC phase (forward passes in fp32). Geometry runs fine on CPU.

In [ ]:
# Cell 5
!nvidia-smi || echo "No GPU visible — geometry still runs, LMC does not."
import psutil
print(f"\nRAM total {psutil.virtual_memory().total/1e9:.1f} GB  "
      f"available {psutil.virtual_memory().available/1e9:.1f} GB")
print("Rule of thumb: pairwise streaming needs ~8 GB free. Below that, lower LMC_BATCH first.")

**Cell 6**

## 3. Install dependencies

Same pinned set as NB08 minus `trl` and `bitsandbytes` (no PPO, no quantised reward model), plus `scipy` for the floor LP. The Transformers / PEFT / Accelerate pins stay because the adapters were written by PEFT 0.10.0 and θ_SFT must be reconstructed by the same loader that produced it in NB08.

In [ ]:
# Cell 7
!pip uninstall -y torchao
!pip install -q -U "pandas==2.2.2" "numpy<2.1" "protobuf>=5.29.1,<6.0.0" "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" scipy datasets pyyaml safetensors psutil

**Cell 8**

Restart the runtime once after installation so Python forgets previously imported Transformers or PyTorch modules, then rerun the repository cell (Cell 3) and continue.

**Cell 9**

## 4. Settings and gates

`RUN_*` flags mirror NB08's gate structure: **each flag is only switched on after the previous phase has written a passing entry into `report.json`.** Do not enable them all at once — the whole point is that a failed gate stops the pipeline before it contaminates the next stage.

Gate order: `RESTORE → PROVENANCE → STATIC → GEOMETRY → FLOOR → LMC`.

`ADAPTER_ZIP` and `ADAPTER_ZIP_SHA256` point at the NB08 bundle in Drive. The SHA is what makes the chain auditable: it is the evidence that the adapters R was computed from are the adapters PPO produced.

In [ ]:
# Cell 10
import os
from pathlib import Path

BASE_MODEL       = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME     = "nvidia/HelpSteer2"
NOTEBOOK_VERSION = "NB09.1r5"
RUN_TAG          = "run1"

# --- inputs -----------------------------------------------------------------
ADAPTER_ZIP        = "/content/drive/MyDrive/rs_ppo_armorm_adapters.zip"
ADAPTER_SHA_FILE   = "/content/drive/MyDrive/rs_ppo_armorm_adapters.sha256"
THETA_SFT_PROV     = "/content/drive/MyDrive/theta_sft_provenance.txt"   # optional revision pin
CONFIG_PATH        = Path("/content/master-thesis/configs/tinyllama_helpsteer2_armorm.yaml")
NOTEBOOK_PATH      = Path("/content/master-thesis/notebooks/09_1_geometry_rs_ppo_adapters_colab.ipynb")

# --- outputs ----------------------------------------------------------------
WORK_DIR    = Path(f"/content/nb09_1_{RUN_TAG}")
RESTORE_DIR = WORK_DIR / "adapters"
THETA_SFT   = WORK_DIR / "theta_sft" / "merged"
ART_DIR     = WORK_DIR / "artifacts"
REPORT      = ART_DIR / "report.json"
REPO_RESULTS = Path(f"/content/master-thesis/results/nb09_1_geometry_{RUN_TAG}")

AXES = ["helpfulness", "correctness", "coherence", "complexity", "verbosity"]

# --- precision: non-negotiable ----------------------------------------------
MERGE_DTYPE = "float32"      # S9/S10 assert this; bf16 destroys the endpoint

# --- geometry settings ------------------------------------------------------
ORACLE_TOL      = 1e-6       # max rel. error between merge.py and the rank-space oracle
FLOOR_SEED      = 137        # same seed as the PPO prompt stream, by convention
N_DIRICHLET     = 64

# --- LMC settings -----------------------------------------------------------
LMC_T_GRID      = [round(k / 20, 2) for k in range(21)]
LMC_N_SEQ       = 256
LMC_MAX_LEN     = 512
LMC_BATCH       = 4
LMC_SEED        = 137
LMC_BOOTSTRAP_N = 2000       # shared paired bootstrap over complete sequences
LMC_CI_ALPHA    = 0.05       # one-sided familywise 95% bounds; pointwise CIs are descriptive
LMC_BARRIER_EPS = 1e-9       # numerical zero for detection, not practical relevance
LMC_EQUIV_MARGIN = None      # nats/token; set ONLY after Lingxiao pre-registers delta
LMC_RESAMPLING_UNIT = "sequence"
LMC_MULTIPLICITY = "shared-resample centered max-deviation"

# --- floor settings ---------------------------------------------------------
FLOOR_PRIMARY_MATRIX     = "R_cos"   # scale-free matrix used by the coefficient rule
FLOOR_SENSITIVITY_MATRIX = "R_gram"  # magnitude-sensitive robustness check
FLOOR_MATRICES = [FLOOR_PRIMARY_MATRIX, FLOOR_SENSITIVITY_MATRIX]
FLOOR_OPEN_TOL   = 1e-9      # strict-opening threshold on the LP value t*
INVERSE_POS_TOL  = 1e-12     # positivity threshold on entries of R^-1 1
PD_REL_TOL       = 1e-10     # eigmin > PD_REL_TOL * eigmax for a usable linear solve
TANGENT_SIGN_TOL = 1e-12     # sign threshold on off-diagonals of P R P

# --- gates: enable one at a time --------------------------------------------
RUN_RESTORE   = True
RUN_STATIC    = False
RUN_GEOMETRY  = False
RUN_FLOOR     = False
RUN_LMC       = False

for d in (WORK_DIR, RESTORE_DIR, ART_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Drive supplies the NB08 adapter bundle and an optional base-revision sidecar.
# Result artifacts are exported for download and for the repository (Cells 26-28).
from google.colab import drive
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Drive not mounted — the adapter bundle cannot be read"

print(f"notebook    = {NOTEBOOK_VERSION}")
print(f"base        = {BASE_MODEL}")
print(f"merge dtype = {MERGE_DTYPE}")
print(f"work        = {WORK_DIR}")
print(f"repo out    = {REPO_RESULTS}")
print(f"gates       = restore:{RUN_RESTORE} static:{RUN_STATIC} geom:{RUN_GEOMETRY} "
      f"floor:{RUN_FLOOR} lmc:{RUN_LMC}")
print(f"floor R     = primary {FLOOR_PRIMARY_MATRIX}; sensitivity {FLOOR_SENSITIVITY_MATRIX}")
print(f"lmc grid    = {len(LMC_T_GRID)} points, {LMC_N_SEQ} sequences, "
      f"bootstrap {LMC_BOOTSTRAP_N}")
print(f"lmc inference = {LMC_MULTIPLICITY}; "
      f"equivalence delta={LMC_EQUIV_MARGIN if LMC_EQUIV_MARGIN is not None else 'UNSET'}")


**Cell 11**

A tiny report helper. Every phase appends its own block and the file is rewritten atomically, so a crashed cell cannot leave a half-written report that the next gate then reads as a pass.

In [ ]:
# Cell 12
import json, os, tempfile
from pathlib import Path

def report_read():
    if REPORT.exists():
        return json.loads(REPORT.read_text())
    return {}

def report_write(phase, payload):
    d = report_read()
    d[phase] = payload
    fd, tmp = tempfile.mkstemp(dir=str(ART_DIR)); os.close(fd)
    Path(tmp).write_text(json.dumps(d, indent=2, sort_keys=True, default=str))
    os.replace(tmp, REPORT)
    print(f"[report] {phase}: {payload.get('status', '?')}")

def gate_passed(phase):
    return report_read().get(phase, {}).get("status") == "PASS"

def require(phase):
    assert gate_passed(phase), f"gate {phase!r} has not passed — do not enable the next flag"

print("report helper ready:", REPORT)

**Cell 13**

## 5. Restore the adapters from the NB08 bundle

The bundle is verified against its SHA256 sidecar **before** extraction, not after. A corrupted download that is unzipped first and checked later leaves a plausible-looking adapter tree on disk that someone will eventually use.

After extraction each axis must contribute all three files: the adapter weights, the value head, and the PPO log. A missing `value_head.pt` is harmless for geometry but signals an incomplete bundle, and an incomplete bundle should not silently become the basis of the R that goes into the thesis.

In [ ]:
# Cell 14
import hashlib, zipfile
from pathlib import Path

if RUN_RESTORE:
    zip_path = Path(ADAPTER_ZIP)
    assert zip_path.exists(), f"bundle not found: {zip_path}"

    h = hashlib.sha256()
    with open(zip_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    print(f"bundle   {zip_path.name}  {zip_path.stat().st_size/1e6:.1f} MB")
    print(f"sha256   {digest}")

    sha_file = Path(ADAPTER_SHA_FILE)
    expected = None
    if sha_file.exists():
        expected = sha_file.read_text().split()[0].strip()
        assert digest == expected, (
            f"SHA256 MISMATCH\n  expected {expected}\n  actual   {digest}\n"
            "The bundle is not the one NB08 produced. Stop.")
        print("sha256   verified against sidecar")
    else:
        print(f"sha256   NO sidecar at {sha_file} — recording digest, chain is weaker")

    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RESTORE_DIR)

    need = ["adapter/adapter_model.safetensors", "value_head.pt", "ppo_log.json"]
    missing, found = [], {}
    for a in AXES:
        for f in need:
            p = RESTORE_DIR / f"ppo_{a}" / f
            if not p.exists():
                missing.append(f"{a}/{f}")
        found[a] = str(RESTORE_DIR / f"ppo_{a}" / "adapter")
    assert not missing, f"bundle incomplete: {missing}"

    ADAPTER_PATHS = {a: Path(found[a]) for a in AXES}
    for a in AXES:
        print(f"  {a:12} {found[a]}")

    report_write("restore", {
        "status": "PASS", "zip": str(zip_path), "sha256": digest,
        "zip_size_bytes": zip_path.stat().st_size,
        "sha256_sidecar": str(sha_file), "expected_sha256": expected,
        "sha256_verified": sha_file.exists(), "axes": AXES,
        "adapter_paths": found,
    })
else:
    ADAPTER_PATHS = {a: RESTORE_DIR / f"ppo_{a}" / "adapter" for a in AXES}
    print("RUN_RESTORE off — reusing previously extracted adapters")

**Cell 15**

## 6. Reconstruct θ_SFT and capture provenance

θ_SFT is the **base shortcut**: an untouched fp32 snapshot of TinyLlama-1.1B-Chat-v1.0, no training of any kind. NB09.1 resolves an immutable Hub revision before downloading, passes that revision to both model and tokenizer, and rejects a cached snapshot whose revision marker differs.

The provenance gate records the base-model revision, adapter-bundle SHA256, experiment-config SHA256, notebook SHA256, repository commit and dirty state, package versions, Python/platform details, and CUDA device in `report.json`. This closes the previously open reconstruction chain before any static or geometric gate may run.


In [ ]:
# Cell 16
import torch
import hashlib, importlib.metadata, platform, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

require("restore")
REVISION = None
prov = Path(THETA_SFT_PROV)
if prov.exists():
    prov_lines = [line.strip() for line in prov.read_text().splitlines() if line.strip()]
    if prov_lines and not prov_lines[0].startswith("revision "):
        assert prov_lines[0] == BASE_MODEL, f"provenance model mismatch: {prov_lines[0]}"
    for line in prov_lines:
        if line.startswith("revision "):
            REVISION = line.split()[1].strip()
    print(f"revision pinned from provenance file: {REVISION}")

if REVISION is None:
    from huggingface_hub import HfApi
    REVISION = HfApi().model_info(BASE_MODEL).sha
    assert REVISION, "could not resolve an immutable base-model revision"
    prov.parent.mkdir(parents=True, exist_ok=True)
    prov.write_text(f"{BASE_MODEL}\nrevision {REVISION}\ndtype float32\n")
    print(f"revision resolved and recorded: {REVISION}")

snapshot_revision = THETA_SFT / "source_revision.txt"
if (THETA_SFT / "config.json").exists():
    assert snapshot_revision.exists(), "cached theta_SFT has no revision marker; delete WORK_DIR and rerun"
    assert snapshot_revision.read_text().strip() == REVISION, "cached theta_SFT revision mismatch"
    print(f"theta_SFT already present at {THETA_SFT}")
else:
    THETA_SFT.mkdir(parents=True, exist_ok=True)
    m = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, revision=REVISION, torch_dtype=torch.float32)
    m.save_pretrained(str(THETA_SFT))
    AutoTokenizer.from_pretrained(BASE_MODEL, revision=REVISION).save_pretrained(str(THETA_SFT))
    snapshot_revision.write_text(REVISION + "\n")
    del m
    print(f"theta_SFT = base snapshot written to {THETA_SFT} (fp32)")


def file_sha256(file_path):
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def git_output(*args):
    return subprocess.check_output(
        ["git", "-C", "/content/master-thesis", *args], text=True).strip()


def pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


git_commit = git_output("rev-parse", "HEAD")
git_status = git_output("status", "--porcelain")
restore_info = report_read()["restore"]
assert restore_info.get("sha256"), "adapter SHA256 missing from restore report"
assert CONFIG_PATH.is_file(), f"missing experiment config: {CONFIG_PATH}"
notebook_source_available = NOTEBOOK_PATH.is_file()
notebook_sha256 = file_sha256(NOTEBOOK_PATH) if notebook_source_available else None

report_write("provenance", {
    "status": "PASS",
    "notebook_version": NOTEBOOK_VERSION, "run_tag": RUN_TAG,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "base_model": BASE_MODEL, "base_model_revision": REVISION,
    "theta_sft_role": "untrained fp32 base-model snapshot (project shortcut)",
    "adapter_zip": restore_info["zip"],
    "adapter_zip_sha256": restore_info["sha256"],
    "adapter_zip_sha256_verified": restore_info["sha256_verified"],
    "config_path": str(CONFIG_PATH), "config_sha256": file_sha256(CONFIG_PATH),
    "notebook_path": str(NOTEBOOK_PATH), "notebook_sha256": notebook_sha256,
    "notebook_source_available": notebook_source_available,
    "repository_commit": git_commit, "repository_dirty": bool(git_status),
    "repository_status_porcelain": git_status.splitlines(),
    "python": sys.version, "platform": platform.platform(),
    "packages": {name: pkg_version(name) for name in
                 ["torch", "transformers", "peft", "accelerate", "numpy", "scipy", "datasets"]},
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})


**Cell 17**

## 7. Static checks S9 / S10

The NB08 verifier reported 16 of 18 checks; S9 and S10 never appeared in the output and the handoff instruction was not to treat that as a PASS. Both belong here, because both guard the path this notebook actually uses.

- **S9 — `merge_linearity_verified` must not be hardcoded.** The flag has to be *derived* from a numerical comparison that can fail, not asserted in source. S9 greps the merge module for a literal assignment and refuses to pass if it finds one.
- **S10 — no reduced precision anywhere in the evaluation path.** `MERGE_DTYPE` must resolve to `torch.float32`, and the merge module must not silently default to bf16.

These run before any geometry. If either fails, nothing downstream is trustworthy.

In [ ]:
# Cell 18
import re, inspect, torch

if RUN_STATIC:
    require("provenance")
    import src.merge as M
    src_text = inspect.getsource(M)
    checks = {}

    # --- S9 -----------------------------------------------------------------
    hardcoded = re.findall(r"merge_linearity_verified\s*=\s*(True|1)\b", src_text)
    checks["S9_no_hardcoded_linearity_flag"] = {
        "pass": len(hardcoded) == 0,
        "detail": f"{len(hardcoded)} literal assignment(s) found",
    }

    # --- S10 ----------------------------------------------------------------
    resolved = M.resolve_torch_dtype(MERGE_DTYPE)
    default  = M.resolve_torch_dtype(None)
    checks["S10_merge_dtype_is_fp32"] = {
        "pass": (MERGE_DTYPE == "float32" and resolved is torch.float32
                 and default is torch.float32),
        "detail": f"MERGE_DTYPE={MERGE_DTYPE} resolved={resolved} default={default}",
    }
    bad_default = re.search(r"dtype\s*[:=]\s*[\"']?(bf16|bfloat16|fp16|float16)", src_text)
    checks["S10b_no_reduced_precision_default"] = {
        "pass": bad_default is None,
        "detail": "clean" if bad_default is None else f"found {bad_default.group(0)!r}",
    }

    for name, c in checks.items():
        print(f"  {'PASS' if c['pass'] else 'FAIL'}  {name}: {c['detail']}")

    ok = all(c["pass"] for c in checks.values())
    report_write("static_checks", {
        "status": "PASS" if ok else "FAIL", "checks": checks})
    assert ok, "S9/S10 failed — do not compute geometry on this merge path"
else:
    print("RUN_STATIC off")

**Cell 19**

## 8. D and R

$D_i$ is the exact effective LoRA update of axis $i$, i.e. $\theta_i - \theta_{\mathrm{SFT}}$ summed over all seven target modules and all layers. $R = D^\top D$ is its Gram matrix; $R_{\cos}$ normalises it to unit diagonal.

**Two independent paths, deliberately.**

1. **Primary — `src/merge.py`.** `effective_deltas` materialises $s_i B_iA_i$ per module in fp32 and R is accumulated from the flattened inner products. This is the production code the rest of the project uses, so it is what must be tested.
2. **Oracle — rank space.** For a single module, $\langle\delta_i,\delta_j\rangle = s_is_j\,\mathrm{tr}\!\left((B_i^\top B_j)(A_jA_i^\top)\right)$, where both factors are $r\times r$ with $r=8$. This never materialises the full update, uses a completely different arithmetic path, and is exact.

Agreement between the two to `ORACLE_TOL` is what sets `merge_linearity_verified` — the flag S9 refuses to see hardcoded.

**And a control that must fail.** A test that cannot fail proves nothing. Cell 20 also computes the PEFT-linear cross-term contamination on the same adapters. If that control comes out *within* tolerance, the oracle is not sensitive enough to detect the very error it exists to catch, and the whole check is void regardless of what the primary comparison says.

Pairs are streamed: at most two adapters are resident at any moment.

In [ ]:
# Cell 20
import numpy as np, gc, itertools, time, torch
from pathlib import Path

if RUN_GEOMETRY:
    require("static_checks")
    from src.merge import effective_deltas
    from src.effective_lora_geometry import load_effective_lora_geometry

    m = len(AXES)
    R_primary = np.zeros((m, m), dtype=np.float64)
    R_oracle  = np.zeros((m, m), dtype=np.float64)
    R_peft    = np.zeros((m, m), dtype=np.float64)   # contaminated control

    GEO = {a: load_effective_lora_geometry(Path(ADAPTER_PATHS[a])) for a in AXES}
    modules = sorted(GEO[AXES[0]].keys())
    print(f"{len(modules)} LoRA target modules per adapter")

    def inner_oracle(a, b):
        # <d_a, d_b> in rank space: tr((Ba^T Bb)(Ab Aa^T)) * s_a * s_b
        tot = 0.0
        for mn in modules:
            la, lb = GEO[a][mn], GEO[b][mn]
            Ba = la.lora_b.detach().to(torch.float64)
            Aa = la.lora_a.detach().to(torch.float64)
            Bb = lb.lora_b.detach().to(torch.float64)
            Ab = lb.lora_a.detach().to(torch.float64)
            P = Ba.T @ Bb                       # r x r
            Q = Ab @ Aa.T                       # r x r
            tot += float((P * Q.T).sum()) * float(la.scaling) * float(lb.scaling)
        return tot

    def inner_peft_control(a, b):
        # one of the spurious Ba@Ab cross terms PEFT's linear path introduces
        tot = 0.0
        for mn in modules:
            la, lb = GEO[a][mn], GEO[b][mn]
            Ba = la.lora_b.detach().to(torch.float64); Aa = la.lora_a.detach().to(torch.float64)
            Bb = lb.lora_b.detach().to(torch.float64); Ab = lb.lora_a.detach().to(torch.float64)
            sa, sb = float(la.scaling), float(lb.scaling)
            d_a = sa * (Ba @ Aa)
            cross = np.sqrt(sa * sb) * (Ba @ Ab)     # one of the spurious terms
            tot += float((d_a * cross).sum())
        return tot

    t0 = time.time()
    for i, j in itertools.combinations_with_replacement(range(m), 2):
        ai, aj = AXES[i], AXES[j]
        paths = {ai: Path(ADAPTER_PATHS[ai])} if i == j else \
                {ai: Path(ADAPTER_PATHS[ai]), aj: Path(ADAPTER_PATHS[aj])}
        dl = effective_deltas(paths)
        val = 0.0
        for mn in modules:
            val += float((dl[ai][mn].to(torch.float64) * dl[aj][mn].to(torch.float64)).sum())
        R_primary[i, j] = R_primary[j, i] = val
        R_oracle[i, j]  = R_oracle[j, i]  = inner_oracle(ai, aj)
        R_peft[i, j]    = R_peft[j, i]    = inner_peft_control(ai, aj)
        del dl; gc.collect()
        print(f"  ({ai[:6]},{aj[:6]}) primary={val:.6e}  oracle={R_oracle[i,j]:.6e}")

    scale = np.abs(R_primary).max()
    rel_err     = np.abs(R_primary - R_oracle).max() / scale
    ctrl_relerr = np.abs(R_peft - R_oracle).max() / scale
    print(f"\nelapsed {time.time()-t0:.0f}s")
    print(f"max rel. error  primary vs oracle : {rel_err:.3e}   (tol {ORACLE_TOL:.0e})")
    print(f"max rel. error  PEFT control      : {ctrl_relerr:.3e}   (must exceed tol)")

    merge_linearity_verified = bool(rel_err < ORACLE_TOL)
    control_has_teeth        = bool(ctrl_relerr > ORACLE_TOL)

    d = np.sqrt(np.diag(R_primary))
    R_cos = R_primary / np.outer(d, d)
    eigvals = np.linalg.eigvalsh(R_primary)
    w, V = np.linalg.eigh(R_primary)
    perron = np.abs(V[:, -1]); perron = perron / perron.sum()

    np.save(ART_DIR / "R_gram.npy", R_primary)
    np.save(ART_DIR / "R_cos.npy", R_cos)
    np.save(ART_DIR / "D_norms.npy", d)

    print("\nR_cos:")
    print("            " + "".join(f"{a[:6]:>9}" for a in AXES))
    for i, a in enumerate(AXES):
        print(f"{a:12}" + "".join(f"{R_cos[i,k]:9.3f}" for k in range(m)))
    print(f"\n||D_i||   : {np.array2string(d, precision=4)}")
    print(f"eigenvalues: {np.array2string(eigvals, precision=4)}")
    print(f"Perron     : {np.array2string(perron, precision=4)}")

    ok = merge_linearity_verified and control_has_teeth
    report_write("geometry", {
        "status": "PASS" if ok else "FAIL",
        "merge_linearity_verified": merge_linearity_verified,
        "control_has_teeth": control_has_teeth,
        "oracle_max_rel_err": rel_err,
        "peft_control_rel_err": ctrl_relerr,
        "oracle_tol": ORACLE_TOL,
        "axes": AXES,
        "D_norms": d.tolist(),
        "R_gram": R_primary.tolist(),
        "R_cos": R_cos.tolist(),
        "eigenvalues": eigvals.tolist(),
        "perron": perron.tolist(),
        "n_modules": len(modules),
    })
    assert merge_linearity_verified, "merge.py disagrees with the oracle — do not proceed"
    assert control_has_teeth, "PEFT control passed the tolerance — the check cannot fail, so it proves nothing"
else:
    print("RUN_GEOMETRY off")

**Cell 21**

## 9. Deliverable (b) — positive-$R$ diagnostics, floor LP, and $R^-$

The measured relationship matrices have strictly positive off-diagonal entries. NB09.1r5 treats this as a result, not as a defect to repair:

- $R^-_{ij}=\max(0,-R_{ij})$ for $i\ne j$, with zero diagonal. Hence $R^-=0$ and $C(\lambda)=\lambda^\top R^-\lambda=0$ when every off-diagonal entry is positive. Sign-conflict and PCGrad-style diagnostics are then inactive.
- Entrywise positivity does **not** imply that the common-ascent floor is open or collapsed. The sign diagnostic and the floor diagnostic are logically independent.
- The tangent-centred matrix $P_\perp R P_\perp$, $P_\perp=I-\frac1m\mathbf1\mathbf1^\top$, is reported only as **relative geometry after removal of the common mode**. Negative entries there are not relabelled as conflicts in the original $R$.

The improvement floor is

$$\mathcal F_p=\{\lambda\in\Delta^{m-1}:R(\lambda-p)\ge0\}.$$

Its strict-opening LP is

$$t^\star(p)=\max_{\lambda\in\Delta^{m-1}} t
\quad\text{s.t.}\quad R(\lambda-p)\ge t\mathbf1.$$

Because $\lambda=p$ is feasible, $t^\star\ge0$. A positive value proves that the floor is open for that preference. A zero value alone does not rule out a non-trivial weak direction, so NB09.1r5 does not call that a global collapse without a certificate.

For symmetric positive-definite $R$, put $u=R^{-1}\mathbf1$. Then

$$K=\{v:\mathbf1^\top v=0,\ Rv\ge0\}=\{0\}
\quad\Longleftrightarrow\quad u>0.$$

This criterion is exact: with $w=Rv\ge0$, the tangent condition is $u^\top w=0$. If $u>0$, this forces $w=0$ and therefore $v=0$. Conversely, positive definiteness gives $\mathbf1^\top u>0$; if one component of $u$ is zero or negative, a non-zero $w\ge0$ can be constructed with $u^\top w=0$.

The **primary** floor verdict uses $R_{\cos}$ because the coefficient rule is scale-free. $R_{\mathrm{gram}}$ is reported as a magnitude-sensitive robustness check. Their verdicts may differ; disagreement is a sensitivity finding, not a program error.

The search set contains five vertices, the uniform point, all 13 pre-registered preferences, and 64 Dirichlet(1) draws at seed 137. Six configured preferences duplicate vertices or the uniform point, yielding $|\mathcal B|=77$ unique points.

**Verdict taxonomy (r5).** Because $R^{-1}\mathbf 1>0$ is *equivalent* to $K=\{0\}$ and not merely sufficient, the negated certificate is informative as well: for $R\succ0$, a non-positive entry of $R^{-1}\mathbf 1$ proves $\mathcal F_p\supsetneq\{p\}$ for **every** $p\in\operatorname{ri}(\Delta_m)$. The certificate tests the *weak* cone $Rv\ge0$ whereas the LP tests *strict* common ascent $t^\star>0$, so the two may legitimately disagree in exactly one direction — a certified non-trivial floor with no strict opening, recorded as `FLOOR_NONTRIVIAL_NO_STRICT_OPENING`. The opposite disagreement is a contradiction and aborts the cell. `FLOOR_UNRESOLVED_WEAK` is now reserved for the case where no certificate is available at all, i.e. $R$ fails the relative positive-definiteness test.

**Reading the centred diagnostic.** For an equicorrelation matrix $a I+b(J-I)$ with $a>b$ one has $P R P=(a-b)P$, whose off-diagonals are all $-(a-b)/m<0$. A perfectly conflict-free $R$ therefore already attains the maximum negative-pair count, and on the measured cosine matrix all $\binom{m}{2}=10$ tangent pairs are expected to be negative. The raw count carries no information about conflict; only `tangent_excess_over_equicorrelation` does.


In [ ]:
# Cell 22
import numpy as np
from scipy.optimize import linprog
from src.experiment_config import (
    get_attribute_order,
    load_experiment_config,
    validate_preference_vectors,
)

if RUN_FLOOR:
    require("geometry")

    cfg = load_experiment_config(CONFIG_PATH)
    assert list(get_attribute_order(cfg)) == AXES, "config axis order differs from NB09.1r5"
    prereg = validate_preference_vectors(cfg)
    assert len(prereg) == 13, f"expected 13 pre-registered preferences, found {len(prereg)}"

    m = len(AXES)
    P, names = [], []

    def add_unique(name, vector, tol=1e-12):
        vector = np.asarray(vector, dtype=np.float64)
        for idx, existing in enumerate(P):
            if np.allclose(vector, existing, rtol=0.0, atol=tol):
                return names[idx]
        P.append(vector)
        names.append(name)
        return name

    for i, axis in enumerate(AXES):
        add_unique(axis, np.eye(m)[i])
    add_unique("uniform", np.full(m, 1.0 / m))
    preference_coverage = {
        name: add_unique(f"preference:{name}", values)
        for name, values in prereg.items()
    }
    n_preregistered_unique = sum(
        mapped.startswith("preference:") for mapped in preference_coverage.values())
    assert len(P) == 13, f"expected 13 unique fixed points before Dirichlet, found {len(P)}"

    rng = np.random.default_rng(FLOOR_SEED)
    for k in range(N_DIRICHLET):
        P.append(rng.dirichlet(np.ones(m)))
        names.append(f"dirichlet_{k:02d}")
    assert len(P) == 77, f"expected the pre-registered 77-point set, found {len(P)}"

    def floor_lp(R, p):
        # max t  s.t.  R(lam-p) >= t*1,  sum lam = 1,  lam >= 0
        c = np.concatenate([np.zeros(m), [-1.0]])
        A_ub = np.hstack([-R, np.ones((m, 1))])
        b_ub = -R @ p
        A_eq = np.concatenate([np.ones(m), [0.0]])[None, :]
        bounds = [(0.0, 1.0)] * m + [(None, None)]
        res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=[1.0],
                      bounds=bounds, method="highs")
        if not res.success:
            return np.nan, None
        return float(res.x[-1]), res.x[:m]

    def analyse(tag):
        """Sign, tangent, certificate, and LP diagnostics for one R."""
        R = np.load(ART_DIR / f"{tag}.npy")
        assert R.shape == (m, m), f"{tag}: expected {m}x{m}, got {R.shape}"
        assert np.allclose(R, R.T, atol=1e-10), f"{tag} is not symmetric"

        offdiag = ~np.eye(m, dtype=bool)
        offdiag_strictly_positive = bool((R[offdiag] > 0.0).all())
        entrywise_nonnegative = bool((R >= 0.0).all())

        # Original sign-conflict diagnostic. Do not substitute centred signs here.
        R_minus = np.maximum(-R, 0.0)
        np.fill_diagonal(R_minus, 0.0)
        n_neg_pairs = int((np.triu(R, k=1) < 0.0).sum())
        sign_conflict_status = (
            "NO_SIGN_CONFLICT_SIGNAL" if n_neg_pairs == 0 else "SIGNED_RELATIONS_PRESENT")

        # Relative/tangent geometry after removing the common coefficient mode.
        P_tangent = np.eye(m) - np.ones((m, m)) / m

        def tangent_negatives(M):
            Mt = P_tangent @ M @ P_tangent
            Mt = 0.5 * (Mt + Mt.T)
            return Mt, int((np.triu(Mt, k=1) < -TANGENT_SIGN_TOL).sum())

        R_tangent, tangent_negative_pairs = tangent_negatives(R)
        tangent_eigenvalues = np.linalg.eigvalsh(R_tangent)

        # Mechanical reference. For an equicorrelation matrix a*I + b*(J - I) with
        # a > b one has P R P = (a - b) * P, whose off-diagonals are all
        # -(a - b)/m < 0. A perfectly conflict-free R therefore already produces
        # the MAXIMUM possible number of negative tangent pairs. Reporting the raw
        # count without this reference invites reading a mechanical artefact of
        # centring as evidence of conflict.
        n_pairs = m * (m - 1) // 2
        a_hat = float(np.mean(np.diag(R)))
        b_hat = float(np.mean(R[offdiag]))
        R_equi = b_hat * np.ones((m, m)) + (a_hat - b_hat) * np.eye(m)
        _, tangent_negative_pairs_equi = tangent_negatives(R_equi)
        tangent_excess = tangent_negative_pairs - tangent_negative_pairs_equi
        tangent_status = ("MECHANICAL_ONLY" if tangent_excess <= 0
                          else "EXCESS_OVER_EQUICORRELATION")

        # Exact collapse criterion for symmetric positive-definite R.
        eigenvalues = np.linalg.eigvalsh(R)
        eigmin = float(eigenvalues.min())
        eigmax = float(eigenvalues.max())
        # Relative, not absolute: eigmin > 1e-12 would pass a numerically
        # singular matrix straight into np.linalg.solve.
        positive_definite = bool(eigmin > PD_REL_TOL * max(eigmax, 1.0))
        if positive_definite:
            inverse_ones = np.linalg.solve(R, np.ones(m))
            inverse_min = float(inverse_ones.min())
            inverse_positive = bool((inverse_ones > INVERSE_POS_TOL).all())
        else:
            inverse_ones = None
            inverse_min = None
            inverse_positive = False

        rows, t_vals, c_vals = [], [], []
        for nm, p in zip(names, P):
            t, lam = floor_lp(R, p)
            C = float(p @ R_minus @ p)
            t_vals.append(t)
            c_vals.append(C)
            moved = np.nan if lam is None else float(np.abs(lam - p).max())
            rows.append({"p": nm, "t_star": t, "C": C, "max_shift": moved})

        t_arr = np.asarray(t_vals, dtype=np.float64)
        n_failed = int(np.isnan(t_arr).sum())
        n_open = int((t_arr > FLOOR_OPEN_TOL).sum())
        contradiction = bool(inverse_positive and n_open > 0)
        assert not contradiction, (
            f"{tag}: inverse-positivity certifies K={{0}}, but the LP reports "
            f"{n_open}/{len(t_arr)} strict openings")

        # R^-1 1 > 0  <=>  K = {v : 1'v = 0, Rv >= 0} = {0}  is an EQUIVALENCE for
        # R > 0, so the negation certifies too: a non-positive entry of R^-1 1
        # proves F_p is strictly larger than {p} for EVERY p in ri(Delta_m).
        # Note the certificate tests the WEAK cone (Rv >= 0) while the LP tests
        # STRICT common ascent (t* > 0). The two can legitimately disagree in one
        # direction -- a certified non-trivial floor with no strict opening -- so
        # only the other direction is a contradiction (asserted above).
        floor_nontrivial_certified = bool(positive_definite and not inverse_positive)
        if inverse_positive:
            verdict = "FLOOR_COLLAPSED_CERTIFIED"
        elif n_open > 0:
            verdict = "FLOOR_OPEN_ON_B"
        elif floor_nontrivial_certified:
            verdict = "FLOOR_NONTRIVIAL_NO_STRICT_OPENING"
        else:
            verdict = "FLOOR_UNRESOLVED_WEAK"

        c_max = float(np.abs(np.asarray(c_vals)).max())
        print(f"\n--- {tag} ---")
        print(f"  off-diagonal > 0  : {offdiag_strictly_positive}")
        print(f"  sign diagnostic   : {sign_conflict_status}; negative pairs={n_neg_pairs}; "
              f"max R-={R_minus.max():.3e}")
        print(f"  centred diagnostic: negative pairs={tangent_negative_pairs}/{n_pairs}; "
              f"equicorrelation reference {tangent_negative_pairs_equi}/{n_pairs}; "
              f"excess {tangent_excess:+d} -> {tangent_status}")
        print(f"                      (relative geometry only; centring makes these "
              f"signs negative mechanically)")
        print(f"  eig min            : {eigmin:.6e}")
        inverse_min_text = "n/a" if inverse_min is None else f"{inverse_min:.6e}"
        print(f"  R^-1 1 > 0         : {inverse_positive}; min={inverse_min_text}")
        print(f"  floor open on B    : {n_open}/{len(t_arr)}; "
              f"t* max={np.nanmax(t_arr):.3e}; median={np.nanmedian(t_arr):.3e}")
        print(f"  max |C(p)|         : {c_max:.3e}")
        if n_failed:
            print(f"  ERROR: {n_failed} LP(s) failed")
        print(f"  verdict            : {verdict}")

        return {
            "status": "FAIL" if n_failed else "PASS",
            "relationship_matrix": tag,
            "verdict": verdict,
            "offdiag_strictly_positive": offdiag_strictly_positive,
            "entrywise_nonnegative": entrywise_nonnegative,
            "sign_conflict_status": sign_conflict_status,
            "positive_definite": positive_definite,
            "eigenvalues": eigenvalues.tolist(), "eig_min": eigmin,
            "n_open": n_open, "n_lp_failed": n_failed, "n_preferences": len(t_arr),
            "t_star_max": float(np.nanmax(t_arr)),
            "t_star_median": float(np.nanmedian(t_arr)),
            "R_neg_pair_count": n_neg_pairs,
            "R_neg_count": 2 * n_neg_pairs,
            "R_neg_max_abs": float(R_minus.max()),
            "C_max_abs": c_max,
            "inverse_ones": None if inverse_ones is None else inverse_ones.tolist(),
            "inverse_positive_certificate": inverse_positive,
            "floor_nontrivial_certified": floor_nontrivial_certified,
            "eig_max": eigmax,
            "tangent_eigenvalues": tangent_eigenvalues.tolist(),
            "tangent_negative_pair_count": tangent_negative_pairs,
            "tangent_pair_total": n_pairs,
            "tangent_negative_pair_count_equicorrelation": tangent_negative_pairs_equi,
            "tangent_excess_over_equicorrelation": tangent_excess,
            "tangent_status": tangent_status,
            "tangent_equicorrelation_fit": {"diag_mean": a_hat, "offdiag_mean": b_hat},
            "tangent_note": (
                "P_tangent R P_tangent removes the common coefficient mode. Its signs "
                "describe relative geometry and are not conflicts in the original R. "
                "For an equicorrelation matrix a*I+b*(J-I) with a>b, P R P = (a-b)*P has "
                "ALL off-diagonals equal to -(a-b)/m < 0, so a conflict-free R already "
                "attains the maximum negative-pair count. Only tangent_excess_over_"
                "equicorrelation > 0 carries information beyond centring."),
            "rows": rows,
        }

    per_matrix = {tag: analyse(tag) for tag in FLOOR_MATRICES}
    primary = per_matrix[FLOOR_PRIMARY_MATRIX]
    sensitivity = per_matrix[FLOOR_SENSITIVITY_MATRIX]
    matrix_agreement = primary["verdict"] == sensitivity["verdict"]
    floor_status = (
        "PASS" if all(result["status"] == "PASS" for result in per_matrix.values())
        else "FAIL")

    exact_floor_returns_p = primary["inverse_positive_certificate"]
    floor_nontrivial = primary["floor_nontrivial_certified"]
    sign_methods_inactive = primary["R_neg_pair_count"] == 0
    mover_must_trade_off = exact_floor_returns_p

    if exact_floor_returns_p:
        exact_floor_status = "RETURN_P_FOR_EVERY_P"
    elif floor_nontrivial:
        # The negated certificate is informative, not a gap in knowledge.
        exact_floor_status = "NONTRIVIAL_FEASIBLE_SET_FOR_EVERY_INTERIOR_P"
    else:
        exact_floor_status = "NOT_GLOBALLY_DETERMINED"

    if mover_must_trade_off:
        mover_status = "MUST_DEGRADE_AT_LEAST_ONE_R_PROXY_AXIS"
    elif floor_nontrivial:
        mover_status = "TRADE_OFF_FREE_MOVEMENT_EXISTS_IN_INTERIOR"
    else:
        mover_status = "NO_GLOBAL_STATEMENT"

    method_implications = {
        "sign_conflict_methods": (
            "INACTIVE_NO_NEGATIVE_PAIRS" if sign_methods_inactive else "SIGN_SIGNAL_AVAILABLE"),
        "exact_floor_methods": exact_floor_status,
        "nontrivial_movers": mover_status,
    }

    print(f"\n{'='*72}")
    print(f"primary verdict ({FLOOR_PRIMARY_MATRIX}): {primary['verdict']}")
    print(f"sensitivity ({FLOOR_SENSITIVITY_MATRIX}): {sensitivity['verdict']}")
    print(f"matrix sensitivity: {'AGREE' if matrix_agreement else 'DIFFER'}")
    print(f"sign-conflict methods: {method_implications['sign_conflict_methods']}")
    print(f"exact floor methods  : {method_implications['exact_floor_methods']}")
    print(f"non-trivial movers   : {method_implications['nontrivial_movers']}")
    print("="*72)

    entry = {
        "status": floor_status,
        "verdict": primary["verdict"],
        "primary_matrix": FLOOR_PRIMARY_MATRIX,
        "sensitivity_matrix": FLOOR_SENSITIVITY_MATRIX,
        "matrices": FLOOR_MATRICES,
        "matrices_agree": matrix_agreement,
        "matrix_sensitivity": "AGREE" if matrix_agreement else "DIFFER",
        "per_matrix": per_matrix,
        "method_implications": method_implications,
        "certificate_note": (
            "For symmetric positive-definite R, R^-1 1 > 0 is equivalent to "
            "K={v:1'v=0,Rv>=0}={0}. This is independent of entrywise positivity."),
        "positive_R_note": (
            "Positive off-diagonal entries imply R-=0 and disable sign-conflict "
            "diagnostics; they do not by themselves decide floor openness."),
        "seed": FLOOR_SEED, "n_dirichlet": N_DIRICHLET,
        "n_preregistered": len(prereg),
        "n_preregistered_unique_additions": n_preregistered_unique,
        "preference_coverage": preference_coverage,
        "search_set_note": (
            "5 vertices + uniform + 13 pre-registered preferences (6 duplicates) "
            "+ 64 Dirichlet = 77 unique points"),
    }
    # Backward-compatible flat fields always refer to the primary cosine matrix.
    entry.update({key: primary[key] for key in (
        "n_open", "n_preferences", "t_star_max", "t_star_median",
        "R_neg_count", "R_neg_pair_count", "R_neg_max_abs", "C_max_abs",
        "inverse_ones", "inverse_positive_certificate", "rows")})
    report_write("floor", entry)
    assert floor_status == "PASS", "one or more floor LPs failed"
else:
    print("RUN_FLOOR off")


**Cell 23**

## 10. Deliverable (d) -- linear mode connectivity

LMC asks whether coefficient-space interpolation is loss-connected. The metric is response-token negative log-likelihood on 256 held-out HelpSteer2 responses -- no ArmoRM and no reward query. Prompt and padding tokens are masked.

For path $k$,

$$B_k=\max_t\left[L_k(t)-\left((1-t)L_k(0)+tL_k(1)\right)\right],
\qquad B_{\max}=\max_{k=1,\ldots,10}B_k.$$

The ten vertex-to-vertex paths use the common 21-point grid $\Delta t=0.05$.

### Two questions, two verdicts

1. **Detection:** Is at least one barrier statistically above numerical zero? A familywise detection requires a simultaneous one-sided lower bound above `LMC_BARRIER_EPS`.
2. **Practical equivalence:** Is the largest barrier small enough for the interpolation family to count as practically LMC? This requires a pre-registered margin $\delta$ and the simultaneous upper bound $U_{95}(B_{\max})<\delta$.

Failure to detect a barrier is not evidence of equivalence. Therefore `LMC_EQUIV_MARGIN=None` deliberately produces `EQUIVALENCE_MARGIN_UNSET`; it cannot produce a positive LMC conclusion. After Lingxiao fixes the substantive tolerance in nats/token, enter it once in Cell 10 before running LMC.

### Multiplicity and pre-registration

The resampling unit is the complete sequence. The notebook uses 256 sequences, 2000 bootstrap replicates, $\alpha=0.05$, and one common `BOOT_IDX` plan across all ten paths. Per-path percentile intervals remain descriptive. Inferential decisions use centred maximum-deviation bootstrap quantiles, which yield simultaneous one-sided bounds and preserve the dependence among paths.

A statistically non-zero barrier may still be practically acceptable when its simultaneous upper bound is below $\delta$. Conversely, an interval crossing $\delta$ is `LMC_INCONCLUSIVE`, not evidence for or against LMC.

The pristine target weights are cached once and every update is written as $\theta_0+\delta\theta$ from that cache, preventing accumulation across interpolation points. A behavioural assertion pins coefficient order to adapter-key order.


In [ ]:
# Cell 24
import torch, gc, itertools, numpy as np
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

if RUN_LMC:
    require("floor")
    from datasets import load_dataset
    from src.merge import effective_deltas, combine_effective_deltas, resolve_base_module

    assert torch.cuda.is_available(), "LMC needs a GPU"
    tok = AutoTokenizer.from_pretrained(str(THETA_SFT))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    ds_full = load_dataset(DATASET_NAME, split="validation")
    dataset_fingerprint = getattr(ds_full, "_fingerprint", None)
    ds = ds_full.shuffle(seed=LMC_SEED).select(range(LMC_N_SEQ))
    selection_fingerprint = getattr(ds, "_fingerprint", None)
    template_fallbacks = []

    def as_ids(value):
        return value.tolist() if isinstance(value, torch.Tensor) else list(value)

    def encode_response_only(ex):
        user_msgs = [{"role": "user", "content": ex["prompt"]}]
        full_msgs = user_msgs + [{"role": "assistant", "content": ex["response"]}]
        try:
            prefix_ids = as_ids(tok.apply_chat_template(
                user_msgs, tokenize=True, add_generation_prompt=True))
            full_ids = as_ids(tok.apply_chat_template(
                full_msgs, tokenize=True, add_generation_prompt=False))
            if full_ids[:len(prefix_ids)] != prefix_ids:
                raise ValueError("chat-template prefix is not a prefix of the full conversation")
            response_ids = full_ids[len(prefix_ids):]
        except Exception as exc:
            template_fallbacks.append(f"{type(exc).__name__}: {exc}")
            prefix_text = f"<|user|>\n{ex['prompt']}</s>\n<|assistant|>\n"
            response_text = f"{ex['response']}</s>"
            prefix_ids = tok(prefix_text, add_special_tokens=True)["input_ids"]
            response_ids = tok(response_text, add_special_tokens=False)["input_ids"]

        assert response_ids, "example has no assistant response tokens"
        if len(prefix_ids) + len(response_ids) > LMC_MAX_LEN:
            response_budget = min(len(response_ids), LMC_MAX_LEN - 1)
            prompt_budget = LMC_MAX_LEN - response_budget
            prefix_ids = prefix_ids[-prompt_budget:] if prompt_budget else []
            if len(response_ids) > response_budget:
                original_last = response_ids[-1]
                response_ids = response_ids[:response_budget]
                if tok.eos_token_id is not None and original_last == tok.eos_token_id:
                    response_ids[-1] = tok.eos_token_id

        input_ids = prefix_ids + response_ids
        labels = [-100] * len(prefix_ids) + response_ids
        assert any(token != -100 for token in labels[1:]), "no shifted response label remains"
        return input_ids, labels

    features = [encode_response_only(ex) for ex in ds]
    padded_len = max(len(ids) for ids, _ in features)
    pad_id = tok.pad_token_id
    enc = {
        "input_ids": torch.tensor([
            ids + [pad_id] * (padded_len - len(ids)) for ids, _ in features]),
        "attention_mask": torch.tensor([
            [1] * len(ids) + [0] * (padded_len - len(ids)) for ids, _ in features]),
        "labels": torch.tensor([
            labels + [-100] * (padded_len - len(labels)) for _, labels in features]),
    }
    n_response_tokens = int((enc["labels"][:, 1:] != -100).sum())
    print(f"{len(features)} held-out responses, padded_len {padded_len}, "
          f"supervised response tokens {n_response_tokens}, "
          f"template fallbacks {len(template_fallbacks)}")

    model = AutoModelForCausalLM.from_pretrained(
        str(THETA_SFT), torch_dtype=torch.float32).to("cuda").eval()
    assert next(model.parameters()).dtype is torch.float32, "S10: eval path must be fp32"

    @torch.no_grad()
    def seq_nll():
        """Per-sequence summed response-token NLL and per-sequence token counts.

        Returning per-sequence quantities (instead of one pooled scalar) is what
        makes the paired bootstrap below possible at zero extra GPU cost.
        """
        sums, ntoks = [], []
        for s in range(0, len(features), LMC_BATCH):
            ids = enc["input_ids"][s:s + LMC_BATCH].to("cuda")
            att = enc["attention_mask"][s:s + LMC_BATCH].to("cuda")
            labels = enc["labels"][s:s + LMC_BATCH].to("cuda")
            logits = model(input_ids=ids, attention_mask=att).logits[:, :-1, :]
            tgt = labels[:, 1:]
            ce = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt.reshape(-1), ignore_index=-100, reduction="none",
            ).view(tgt.shape)                      # zero at ignored positions
            sums.append(ce.sum(dim=1).double().cpu())
            ntoks.append((tgt != -100).sum(dim=1).double().cpu())
            del logits, ce
        s_sum = torch.cat(sums).numpy()
        s_ntok = torch.cat(ntoks).numpy()
        assert s_ntok.sum() > 0, "no supervised response tokens"
        return s_sum, s_ntok

    ts = np.array(LMC_T_GRID)

    def barrier_of(curve):
        chord = curve[0] + (curve[-1] - curve[0]) * ts
        return float((curve - chord).max())

    # One fixed bootstrap resample plan, shared by every path so the pairwise
    # barriers are directly comparable.
    boot_rng = np.random.default_rng(LMC_SEED)
    BOOT_IDX = boot_rng.integers(0, len(features),
                                 size=(LMC_BOOTSTRAP_N, len(features)))
    ci_lo_pct, ci_hi_pct = 100 * LMC_CI_ALPHA / 2, 100 * (1 - LMC_CI_ALPHA / 2)

    lmc_rows = []
    lmc_boot = []
    for i, j in itertools.combinations(range(len(AXES)), 2):
        ai, aj = AXES[i], AXES[j]
        DL = effective_deltas({ai: Path(ADAPTER_PATHS[ai]), aj: Path(ADAPTER_PATHS[aj])})
        mods = sorted(DL[ai].keys())
        handles = {mn: resolve_base_module(model, mn) for mn in mods}
        cache = {mn: handles[mn].weight.detach().clone().cpu() for mn in mods}

        # Behavioural test of the coefficient/key contract: coefficient 1.0 must
        # land on the FIRST key of DL. An order mismatch would silently swap
        # t and 1-t (harmless for the barrier, wrong for the reported curve).
        assert list(DL.keys()) == [ai, aj], \
            f"effective_deltas reordered keys: {list(DL.keys())} != {[ai, aj]}"
        probe = combine_effective_deltas([1.0, 0.0], DL)
        assert torch.equal(probe[mods[0]], DL[ai][mods[0]]), \
            "combine_effective_deltas does not zip coefficients with DL key order"
        del probe

        S = np.zeros((len(LMC_T_GRID), len(features)), dtype=np.float64)
        ntok_ref = None
        for k, t in enumerate(LMC_T_GRID):
            merged = combine_effective_deltas([1.0 - t, t], DL)
            with torch.no_grad():
                for mn in mods:
                    w = handles[mn].weight
                    w.copy_((cache[mn] + merged[mn]).to(w.device, torch.float32))
            s_sum, s_ntok = seq_nll()
            if ntok_ref is None:
                ntok_ref = s_ntok
            else:
                assert np.array_equal(ntok_ref, s_ntok), \
                    "token counts drifted across the path -- data is not held fixed"
            S[k] = s_sum
            del merged
        losses = (S.sum(axis=1) / ntok_ref.sum()).tolist()

        with torch.no_grad():
            for mn in mods:
                handles[mn].weight.copy_(cache[mn].to(handles[mn].weight.device))
        del DL, cache
        gc.collect()
        torch.cuda.empty_cache()

        L = np.array(losses)
        barrier = barrier_of(L)

        # Paired bootstrap over sequences: the whole 21-point curve is recomputed
        # from the same resampled sequence set, so the barrier -- a max over the
        # path -- inherits a valid CI.
        boot = np.empty(LMC_BOOTSTRAP_N)
        for b in range(LMC_BOOTSTRAP_N):
            idx = BOOT_IDX[b]
            boot[b] = barrier_of(S[:, idx].sum(axis=1) / ntok_ref[idx].sum())
        lo, hi = (float(x) for x in np.percentile(boot, [ci_lo_pct, ci_hi_pct]))
        se = float(boot.std(ddof=1))
        pointwise_detected = bool(lo > LMC_BARRIER_EPS)
        lmc_boot.append(boot.copy())  # replicate b is shared across every path

        lmc_rows.append({"pair": f"{ai}|{aj}", "t_grid": LMC_T_GRID,
                         "loss": L.tolist(), "barrier": barrier,
                         "pointwise_ci_lo": lo, "pointwise_ci_hi": hi,
                         "barrier_se": se,
                         "pointwise_detected_uncorrected": pointwise_detected,
                         "t_argmax": float(ts[int(np.argmax(L - (L[0] + (L[-1] - L[0]) * ts)))]),
                         "n_response_tokens": float(ntok_ref.sum())})
        print(f"  {ai[:6]:>6}-{aj[:6]:<6} L[0]={L[0]:.4f} L[1]={L[-1]:.4f}  "
              f"barrier={barrier:+.6f}  pointwise CI[{lo:+.6f}, {hi:+.6f}]  "
              f"SE={se:.6f}  "
              f"{'POINTWISE' if pointwise_detected else 'not pointwise'}")

    barriers = np.array([row["barrier"] for row in lmc_rows], dtype=np.float64)
    boot_matrix = np.stack(lmc_boot, axis=1)  # [bootstrap replicate, path]
    assert boot_matrix.shape == (LMC_BOOTSTRAP_N, len(lmc_rows))
    assert len(lmc_rows) == 10, f"expected ten vertex paths, got {len(lmc_rows)}"

    def simultaneous_lmc_intervals(point, boot, alpha):
        """One-sided familywise bounds from a centred max-deviation bootstrap."""
        point = np.asarray(point, dtype=np.float64)
        boot = np.asarray(boot, dtype=np.float64)
        assert boot.ndim == 2 and boot.shape[1] == point.size
        assert 0.0 < alpha < 1.0
        centred = boot - point[None, :]
        # Clamp at zero so the reported confidence set always contains the
        # point estimate; this is conservative when bootstrap bias would make a
        # basic one-sided bound cross the estimate.
        q_lower = max(0.0, float(np.quantile(
            np.max(centred, axis=1), 1.0 - alpha)))
        q_upper = max(0.0, float(np.quantile(
            np.max(-centred, axis=1), 1.0 - alpha)))
        lower = np.maximum(0.0, point - q_lower)
        upper = point + q_upper
        return lower, upper, q_lower, q_upper

    sim_lo, sim_hi, q_lo, q_hi = simultaneous_lmc_intervals(
        barriers, boot_matrix, LMC_CI_ALPHA)
    assert np.all(sim_lo <= barriers + 1e-15)
    assert np.all(sim_hi >= barriers - 1e-15)

    for row, lo, hi in zip(lmc_rows, sim_lo, sim_hi):
        row["simultaneous_lower"] = float(lo)
        row["simultaneous_upper"] = float(hi)
        row["familywise_detected"] = bool(lo > LMC_BARRIER_EPS)

    ses = np.array([row["barrier_se"] for row in lmc_rows])
    pointwise_n_detected = int(sum(
        row["pointwise_detected_uncorrected"] for row in lmc_rows))
    familywise_n_detected = int(sum(row["familywise_detected"] for row in lmc_rows))
    bmax_hat = float(barriers.max())
    bmax_lower = float(sim_lo.max())
    bmax_upper = float(sim_hi.max())

    detection_verdict = (
        "FAMILYWISE_BARRIER_DETECTED" if familywise_n_detected > 0
        else "NO_FAMILYWISE_BARRIER_DETECTED")

    if LMC_EQUIV_MARGIN is None:
        equivalence_verdict = "EQUIVALENCE_MARGIN_UNSET"
        margin = None
    else:
        margin = float(LMC_EQUIV_MARGIN)
        assert np.isfinite(margin) and margin > 0.0, \
            "LMC_EQUIV_MARGIN must be a positive finite number in nats/token"
        if bmax_upper < margin:
            equivalence_verdict = "PRACTICALLY_LMC"
        elif bmax_lower > margin:
            equivalence_verdict = "PRACTICALLY_RELEVANT_BARRIER"
        else:
            equivalence_verdict = "LMC_INCONCLUSIVE"

    print(f"\nmax barrier estimate {bmax_hat:+.6f}; "
          f"simultaneous one-sided bounds [{bmax_lower:+.6f}, {bmax_upper:+.6f}]")
    print(f"median bootstrap SE {np.median(ses):.6f}")
    print(f"uncorrected pointwise detections: {pointwise_n_detected}/{len(barriers)}")
    print(f"familywise detections          : {familywise_n_detected}/{len(barriers)}")
    print(f"detection verdict              : {detection_verdict}")
    if margin is None:
        print("equivalence verdict            : EQUIVALENCE_MARGIN_UNSET")
        print("  -> Set delta only after the scientific tolerance has been pre-registered.")
    else:
        print(f"equivalence margin delta       : {margin:.6f} nats/token")
        print(f"equivalence verdict            : {equivalence_verdict}")

    report_write("lmc", {
        "status": "PASS",
        "verdict": equivalence_verdict,
        "detection_verdict": detection_verdict,
        "equivalence_verdict": equivalence_verdict,
        "equivalence_margin_nats_per_token": margin,
        "max_barrier": bmax_hat,
        "max_barrier_simultaneous_lower": bmax_lower,
        "max_barrier_simultaneous_upper": bmax_upper,
        "mean_barrier": float(barriers.mean()),
        "inference_method": LMC_MULTIPLICITY,
        "detection_rule": "any familywise simultaneous lower bound > numerical zero",
        "equivalence_rule": "simultaneous upper bound for maximum barrier < pre-registered delta",
        "simultaneous_q_lower": q_lo, "simultaneous_q_upper": q_hi,
        "bootstrap_n": LMC_BOOTSTRAP_N, "ci_alpha": LMC_CI_ALPHA,
        "ci_sidedness": "one-sided 1-alpha simultaneous bounds",
        "resampling_unit": LMC_RESAMPLING_UNIT,
        "shared_resample_plan_across_paths": True,
        "barrier_eps": LMC_BARRIER_EPS,
        "median_barrier_se": float(np.median(ses)),
        "pointwise_n_detected_uncorrected": pointwise_n_detected,
        "familywise_n_detected": familywise_n_detected,
        "n_paths": len(lmc_rows),
        "dataset_name": DATASET_NAME, "dataset_split": "validation",
        "dataset_fingerprint": dataset_fingerprint,
        "selection_fingerprint": selection_fingerprint,
        "n_sequences": LMC_N_SEQ, "max_len": LMC_MAX_LEN, "seed": LMC_SEED,
        "n_response_tokens": n_response_tokens, "prompt_tokens_masked": True,
        "chat_template_fallback_count": len(template_fallbacks),
        "metric": "mean response-token NLL on held-out HelpSteer2 validation responses (no ArmoRM)",
        "t_grid": LMC_T_GRID,
        "rows": lmc_rows,
    })
    del model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("RUN_LMC off")


**Cell 25**

## 11. Export the results

Nothing here goes to Drive. Two destinations instead:

- **`results/nb09_1_geometry_<tag>/` inside the repository** — the small, text-readable artifacts meant to be committed: `report.json`, the R matrices as CSV, and the SHA256 manifest. Together these are a few kilobytes.
- **a zip download** — the same files bundled, for local archiving.

R is a 5×5 matrix and `report.json` a few hundred lines, so this is exactly the kind of result that belongs under version control: reviewable in a diff, and carrying the adapter bundle's SHA256 so any later claim can be traced back to the PPO run it came from.

The `.npy` files are kept for downstream notebooks, and the same matrices are written as CSV so they are readable in a pull request without loading NumPy.

In [ ]:
# Cell 26
import hashlib, os, shutil, csv
import numpy as np

d = report_read()
required_gates = ["restore", "provenance", "static_checks", "geometry", "floor", "lmc"]
print("gate status:")
for k in required_gates:
    print(f"  {k:15} {d.get(k, {}).get('status', '-')}")
incomplete = [k for k in required_gates if d.get(k, {}).get("status") != "PASS"]
assert not incomplete, f"refusing to export; incomplete gates: {incomplete}"

# --- text-readable exports so a diff shows the actual numbers -----------------
if (ART_DIR / "R_gram.npy").exists():
    for name in ["R_gram", "R_cos"]:
        Mx = np.load(ART_DIR / f"{name}.npy")
        with open(ART_DIR / f"{name}.csv", "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["adapter"] + AXES)
            for a, row in zip(AXES, Mx):
                w.writerow([a] + [f"{v:.12e}" for v in row])
    dn = np.load(ART_DIR / "D_norms.npy")
    with open(ART_DIR / "D_norms.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["axis", "norm"])
        for a, v in zip(AXES, dn):
            w.writerow([a, f"{v:.12e}"])
    print("CSV exports written")

# --- manifest ----------------------------------------------------------------
manifest = []
for f in sorted(ART_DIR.rglob("*")):
    if f.is_file() and f.name != "manifest.sha256":
        manifest.append(f"{hashlib.sha256(f.read_bytes()).hexdigest()}  {f.relative_to(ART_DIR)}")
(ART_DIR / "manifest.sha256").write_text("\n".join(manifest) + "\n")
print(f"{len(manifest)} artifacts hashed")

# --- copy into the repository -------------------------------------------------
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
for f in sorted(ART_DIR.iterdir()):
    if f.is_file():
        shutil.copy2(f, REPO_RESULTS / f.name)
print(f"\ncopied to {REPO_RESULTS}:")
for f in sorted(REPO_RESULTS.iterdir()):
    print(f"  {f.name:24} {f.stat().st_size:>8} B")

# --- zip for download ---------------------------------------------------------
out_zip = f"/content/nb09_1_geometry_{RUN_TAG}.zip"
shutil.make_archive(out_zip[:-4], "zip", root_dir=str(ART_DIR))
print(f"\nzip {os.path.getsize(out_zip)/1e3:.1f} kB  sha256 "
      f"{hashlib.sha256(open(out_zip,'rb').read()).hexdigest()}")

**Cell 27**

## 12. Download

In [ ]:
# Cell 28
import os
out_zip = f"/content/nb09_1_geometry_{RUN_TAG}.zip"
assert os.path.exists(out_zip), f"{out_zip} not found — run Cell 26 first"
try:
    from google.colab import files
    files.download(out_zip)
except ImportError:
    print("Automatic download is available when this notebook runs in Google Colab.")

**Cell 29**

## 13. Git safety check

Commit the contents of `results/nb09_1_geometry_<tag>/` — they are small text artifacts and the whole point is that the numbers are reviewable.

Keep out of the repository, as in the other notebooks: the zip archive, the adapter bundle, `theta_sft/`, and anything under `/content/nb09_1_<tag>/` other than the artifacts folder — i.e. no `.safetensors`, no `.bin`, no checkpoints, no model weights.

`git status` below should show only the new results folder. If it shows adapters or a zip, do not stage everything at once.

In [ ]:
# Cell 30
%cd /content/master-thesis
!git status --short
print("\n--- untracked files larger than 1 MB (should be empty) ---")
!git status --porcelain --untracked-files=all | awk '$1=="??"{print $2}' | while read f; do [ -f "$f" ] && [ $(stat -c%s "$f") -gt 1000000 ] && echo "$f $(stat -c%s "$f") B"; done; true

**Cell 31**

## 14. What comes next

Deliverables (b) and (d) are now on the PPO adapters rather than the v5 SFT-R. NB09.1 includes the complete 77-point floor set, response-only LMC, and auditable provenance.

Still open before the broader study is final:

1. **Method outputs on the new R** — Avg, MaxMin(c), Cert, Fair(α,ε) — should use `R_cos.npy` as the primary scale-free matrix, with `R_gram.npy` as a sensitivity check, and can be computed geometrically without touching ArmoRM: which λ each returns, whether it moves off p, and whether its guarantee is non-empty.
2. **Deliverable (a) stays blocked as independent evidence.** The vertex matrix M, $U_p(\lambda)$ and the method comparison use ArmoRM, which also trained these adapters. They may be reported only as circular upper-bound diagnostics unless an independent evaluator is introduced.
